In [7]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("..\\data\\raw\\movies.csv")
clustered = pd.read_csv("..\\outputs\\clustered_movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
df = movies.merge(clustered,on='movieId')
print(clustered.columns)
df.head()

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', '15', '16', '17', '18', '19', '20', 'movieId', 'group'],
      dtype='object')


,movieId,title,genres,0,1,2,3,4,5,6,...,12,13,14,15,16,17,18,19,20,group
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,6.923529,-0.622405,3.653590,1.650460,-1.074854,1.560218,-0.134682,...,0.514095,-0.255219,-2.794391,3.825341,1.698827,-0.534725,-1.506476,1.326312,0.257197,1
1,2,Jumanji (1995),Adventure|Children|Fantasy,4.497216,-0.043198,2.216751,0.859241,-0.089031,0.363486,0.425357,...,0.896990,1.690563,-2.313469,2.900070,0.253680,-0.885600,-0.727571,-0.798917,-0.147642,1
2,3,Grumpier Old Men (1995),Comedy|Romance,-0.090674,-1.454600,-0.192217,-0.157205,-1.862521,0.130831,-0.317642,...,0.379521,0.086895,0.210605,0.217335,1.022009,-0.619251,-0.426783,0.244291,0.081599,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,-0.912260,-1.702799,-0.356695,-0.120277,-1.836550,-0.063194,0.801612,...,0.012137,0.239836,0.542260,0.121233,0.590657,-0.550151,-0.375454,0.141644,-0.047632,0
4,5,Father of the Bride Part II (1995),Comedy,0.190519,-0.716639,-0.589899,-0.174251,-0.881254,0.279559,-0.878279,...,0.356678,-0.479864,-0.481378,1.107290,-0.227961,0.169393,-0.031123,0.354242,0.365274,0


In [9]:
def recommend(movie_name,n=5):

    movie = df[df['title'].str.contains(movie_name,case=False)] #case=False works with lowercase and uppercase also initcap
    if movie.empty:
        print("Movie not found")
        return
    
    movie_id = movie.iloc[0]['movieId'] #extracting the movieId of the first movie that matches the name
    cluster = movie.iloc[0]['group'] #extracting the cluster of the first movie that matches the name
    cluster_movies = df[df['group']==cluster] #selecting all movies in the same cluster
    features = df.columns.difference(['movieId', 'title', 'genres', 'group']) #selecting all columns except the ones specified
    similarity = cosine_similarity(cluster_movies[features]) #calculating cosine similarity between movies in the same cluster

    idx = cluster_movies.index.get_loc(movie.index[0]) #getting the index of the movie in the cluster_movies dataframe
    scores = list(enumerate(similarity[idx])) #enumerate returns a tuple of index and value, so we are creating a list of tuples of index and similarity score
    scores = sorted(scores,key=lambda x:x[1],reverse=True) #sorting the list of tuples based on the similarity score in descending order

    recommendations = []
    for i,_ in scores[1:n+1]: #skipping the first movie as it is the same movie and getting the next n movies cosine similarity scores with itself = 1.0
        recommendations.append(cluster_movies.iloc[i]['title']) #extracting the title of the movie that are most similar to the movie that was passed as an argument to the function
    return recommendations

In [10]:
recommend("godfather")

['Godfather: Part II, The (1974)',
 'Goodfellas (1990)',
 'American History X (1998)',
 'Green Mile, The (1999)',
 'Catch Me If You Can (2002)']